<div style="
  background-color:#ffffff;
  color:#1f2937;
  padding:28px 24px;
  border-radius:14px;
  border:1px solid #d1d5db;
  font-family:'Segoe UI', Arial, sans-serif;
  text-align:center;
  box-shadow:0 6px 18px rgba(0,0,0,0.06);
">

  <img src="../Su.svg.png" alt="Sorbonne Université Logo" style="
    width:95px;
    height:auto;
    margin-bottom:16px;
  ">

  <div style="
    display:inline-block;
    font-size:22px;
    font-weight:700;
    color:#111827;
    padding-bottom:8px;
    margin-bottom:16px;
  ">
    A-Abo
  </div>

  <div style="font-size:18px; font-weight:600; color:#374151;">
    Projet académique de bases de données
  </div>

  <div style="font-size:15px; margin-top:6px; color:#1e3a8a; font-weight:600;">
    Sorbonne Université — Année universitaire 2025–2026
  </div>

  <div style="
    margin:18px auto 0 auto;
    max-width:650px;
    font-size:12px;
    line-height:1.5;
    color:#6b7280;
    font-style:italic;
  ">
    Ce document est fourni à titre pédagogique et peut faire l’objet de corrections ou d’améliorations.
  </div>

</div>

<center> <b> <h1> TD 5 et TME5 : JOINTURES ET IMBRICATION AVEC IN ET EXISTS </h1>
<center></center> Ce TD/TME utilise les données contenues dans le fichier **bd_jo_v2_H2.sql**


<center> <b> Installation H2 

<center>Le système utilisé pendant les TME est H2.

Télécharger le pilote de H2

In [1]:
pip install psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


https://www.psycopg.org/docs/
http://localhost:8082

**Relancez le kernel**: Kernel -> Restart kernel ...

In [2]:
import psycopg2
import os
local_dir = "./data"
os.makedirs(local_dir, exist_ok=True)
os.listdir(local_dir)

[]

**Attention**: vous pouvez cliquer sur les 3 lignes dans la fenêtre de gauche pour d'afficher les différentes sections du notebook   

# Démarrage du serveur H2
Démarrer un serveur de base de données H2 en arrière-plan, accessible uniquement localement(localhost, 127.0.0.1) sur le port 8082, avec les fichiers des bases de données stockés dans ./data (un fichier *.mv.db par base de données)

In [3]:
port = 8082
print(f'Le numero du port utilisé pour la connexion à la BD est: {port}')

Le numero du port utilisé pour la connexion à la BD est: 8082


In [4]:
%%bash --bg --out output -s "$port"
java -Dh2.bindAddress=127.0.0.1 -cp h2-2.1.214.jar:postgresql-42.6.0.jar org.h2.tools.Server -pg -pgPort $1 -baseDir ./data -ifNotExists

Couldn't find program: 'bash'


Tester si le serveur a été démarré

In [5]:
%%bash
nc -zv 127.0.0.1 8082

Couldn't find program: 'bash'


<span style="color:red"> 
<center> <h1>ADD THIS PART FOR WINDOWS !!

In [6]:
import psycopg2
import os
import socket
import subprocess
import time

In [7]:
local_dir = "./data"
os.makedirs(local_dir, exist_ok=True)
os.listdir(local_dir)

[]

In [8]:
port = 8082
print(f"Le numero du port utilisé pour la connexion à la BD est: {port}")

Le numero du port utilisé pour la connexion à la BD est: 8082


In [102]:
cmd = [
    "java",
    "-Dh2.bindAddress=127.0.0.1",
    "-cp",
    "h2-2.1.214.jar;postgresql-42.6.0.jar",
    "org.h2.tools.Server",
    "-pg",
    "-pgPort",
    str(port),
    "-baseDir",
    "./data",
    "-ifNotExists"
]

h2_process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

time.sleep(2)

if h2_process.poll() is None:
    print("H2 server started successfully in background.")
else:
    out, err = h2_process.communicate()
    print("H2 server failed to start.")
   # print("STDOUT:\n", out)
    #print("STDERR:\n", err)
     # cause Already running check connection = connect_H2( .... ) execute the function H2 and the line after it 

H2 server failed to start.


In [10]:
s = socket.socket()
result = s.connect_ex(("127.0.0.1", port))
s.close()

if result == 0:
    print(f"Server is running on port {port}")
else:
    print(f"Server is NOT running on port {port}")

Server is running on port 8082


# Connexion du client au serveur H2
Se connecter au serveur H2 précédemment lancé en utilisant un nom de base de données, un login et un mot de passe. Si une connexion existante existe, elle est fermée avant d’en créer une nouvelle. Si la base n’existe pas encore, H2 la crée automatiquement dans le répertoire de données ./data (fichier *.mv.db). La fonction retourne un objet de connexion utilisable pour exécuter des commandes SQL sur le serveur.

In [11]:
def connect_H2(db,user,password):
    global connection
    try:
        connection
    except:
        connection = None
    if connection != None:
        try:
            connection.close()
            print("Connection closed")
        except  Error as e:
            print(f"The error '{e}' occurred")
    try:
        connection = psycopg2.connect(f"dbname={db} user={user} password={password} host=localhost port={port}")
        print("Connection to H2 DB successful")
    except Exception as e:
        print(f"The error '{e}' occurred")
    return connection

In [12]:
#Connexion à la base H2 nommée "jo<port>" avec l’utilisateur et le mot de passe spécifiés. Si la base n’existe pas encore, H2 la crée automatiquement. 
#Toute connexion précédente est fermée avant d’en établir une nouvelle.
connection = connect_H2("jo"+f'{port}',"user","pass")

Connection to H2 DB successful


# Fonction execute

Exécute une commande SQL sur la connexion fournie, affiche le nombre de lignes affectées et, si demandé, présente les résultats sous forme de tableau lisible avec les noms de colonnes. Le curseur peut être fermé automatiquement après l’exécution, et toute erreur est affichée. La fonction retourne le curseur utilisé pour accéder aux résultats ou None en cas d’échec.


In [13]:
def execute(connection, query, show=True, close=True, top=10):
    try:
        cursor = connection.cursor()
        cursor.execute(query)
        if cursor.rowcount >0:
          print(cursor.rowcount,"rows,", f"display only {top} rows")
        else:
          print(cursor.rowcount,"rows")
        if show and cursor.rowcount:
            names = [desc[0] for desc in cursor.description]

            lengths={}
            for attr in names:
                lengths[attr]=len(attr)
            for ligne in cursor:
                i=0
                for attr in ligne:
                    lengths[names[i]]=max(lengths[names[i]],len(str(attr)))
                    i=i+1
            print('|',end='')
            for attr in names:
                print(str(attr).ljust(lengths[attr]),end='|')
            print()
            print('|',end='')
            for attr in names:
                print(''.ljust(lengths[attr]+1,'-'),end='')
            print()
            cursor.execute(query)
            n=0
            for ligne in cursor:
                i=0
                print('|',end='')
                for attr in ligne:
                    print(str(attr)[:lengths[names[i]]].ljust(lengths[names[i]]),end='|')
                    i+=1
                print()
                n+=1
                if n>top :
                  break
        if close:
            cursor.close()
    except Exception as e:
        print(f"The error '{e}' occurred")
        cursor=None
    return cursor
print('fonction définie')

fonction définie


# Affichage du schéma d'une table

In [14]:
def show_table(connection,table_name):

    print(f'*********** Attributs de la table : {table_name} ************')
    query=f"""
        SELECT table_name,
              column_name,
              data_type,
              column_default,
              is_nullable,
              character_maximum_length,
              numeric_precision
        FROM information_schema.columns
        where lower(table_name)  = '{table_name}'
        """
    execute(connection,query)


    print(f'\n*********** Contraintes de la table : {table_name} ************')
    query = f"""
        SELECT tc.constraint_name, 
               tc.constraint_type, 
               cc.check_clause
        FROM information_schema.table_constraints tc
        LEFT JOIN information_schema.check_constraints cc 
               ON tc.constraint_name = cc.constraint_name
        WHERE lower(tc.table_name) = '{table_name}'
        AND tc.table_schema = 'public'
        ORDER BY tc.constraint_type
        """
    execute(connection, query)

# Affichage du schéma global 

In [15]:
def show_schema(connection):

    print('*********** Tables ************')
    query="""
    select TABLE_NAME
    from INFORMATION_SCHEMA.TABLES
    where TABLE_SCHEMA = 'public'
    """
    execute(connection,query,show=True,top=1000)

    print('\n\n*********** Domaines ************')
    query="""
    SELECT domain_name,check_clause
    FROM information_schema.domain_constraints a left outer join
         information_schema.check_constraints b
         on a.constraint_name=b.constraint_name
    """
    execute(connection,query,top=1000)

    print('\n\n*********** Attributs ************')
    query=f"""
    SELECT c.table_name,
           c.column_name,
           c.data_type,
           c.column_default,
           c.is_nullable,
           c.character_maximum_length,
           c.numeric_precision
    FROM INFORMATION_SCHEMA.TABLES t, information_schema.columns c
    where t.table_name=c.table_name
    and t.TABLE_SCHEMA = 'public'
    """
    execute(connection,query,top=1000)

    print('\n\n*********** Contraintes d''Intégrité ************')
    query="""
    SELECT 
        tc.table_name, 
        tc.constraint_name, 
        tc.constraint_type,
        cc.check_clause
    FROM information_schema.table_constraints tc
    LEFT JOIN information_schema.check_constraints cc 
        ON tc.constraint_name = cc.constraint_name
    WHERE tc.table_schema = 'public'
    ORDER BY tc.table_name, tc.constraint_type
    """
    execute(connection,query,top=1000)

# TD5 

On considère le schéma de la base JEUXOLYMPIQUE2014 qui décrit les athlètes et leurs résultats aux
épreuves des sports des Jeux Olympiques d'Hiver Sotchi 2014 :

**PAYS** ( <u>CODEPAYS</u>, NOMP)<br/>
Ex. ('FRA', 'France')<br/>
**SPORT** ( <u>SID</u>, NOMSP)<br/>
Ex. (1, 'Biathlon')<br/>
**EPREUVE** ( <u>EPID</u>, SID*, NOMEP, CATÉGORIE, DATEDEBUT, DATEFIN)<br/>
Ex. (10, 1, 'relais 4x7,5km', 'Hommes', 22/02/2014, 22/02/2014)<br/>
**ATHLETE** ( <u>AID</u>, NOMATH, PRENOMATH, DATENAISSANCE, CODEPAYS*)<br/>
Ex. (1000, 'SOBOLEV', 'Alexey', NULL, 'RUS')<br/>
**EQUIPE** ( <u>EQID</u>, CODEPAYS*)<br/>
Ex. (30, 'SUI')<br/>
**ATHLETESEQUIPE** ( <u>EQID*, AID*</u>)<br/>
Ex. (30, 796) : L'athlète (aid=796) a participé à l'équipe (eqid=30)<br/>
**RANGINDIVIDUEL** ( <u>EPID*, AID*</u>, RANG)<br/>
Ex. (15, 61, 1) : L'athlète (aid=61) a gagné la médaille d'or (rang=1) de l'épreuve (epid=15)<br/>
**RANGEQUIPE** ( <u>EPID*, EQID*</u>, RANG)<br/>
Ex. (10, 30, 14) : L'équipe (eqid=30) a été classée 14e à l'épreuve (epid=10)<br/>


Les attributs qui forment la clé primaire de chaque relation sont soulignés. Les clés étrangères sont
signalées avec une *. Les attributs aid, epid, eqid et sid correspondent aux identifiants des athlètes,
épreuves, équipes et sports et sont utilisés à la fois comme clé primaire ou comme référence (clé
étrangère) vers la relation correspondante.
La relation **PAYS** contient le code et le nom de tous les pays, même si ils n'ont pas participé aux
Jeux Olympiques. Les sports (n-uplets de la relation **SPORT**) sont un ensemble d'épreuves (n-uplets
de la relation **EPREUVE**). Pour chaque épreuve on connaît son nom et les date de début et fin de
l'épreuve. Les épreuves peuvent être individuelles ou par équipe. Dans le premier cas, la
participation des athlètes (n-uplets de la relation ATHLETE) est stocké dans la table
**RANGINDIVIDUEL** qui contient en plus le rang qu'ils ont obtenu (1 pour la médaille d'or). Pour les
épreuves par équipe les résultats sont stockés dans la relation **RANGEQUIPE**, alors que l'information
sur le pays de chaque équipe et ses participants et stocké dans les relations **EQUIPE** et
**ATHLETESEQUIPE**. Dans les relations **RANGINDIVIDUEL** et **RANGEQUIPE** l'attribut rang est égal à
null si l'athlète ou l'équipe a été disqualifié.


## Créer les tables et charger les données bd_jo_v2_H2.sql

In [16]:
schemafile=open("bd-jo-v2_H2.sql",mode="r",encoding='utf-8')
create_schema=schemafile.read()
execute(connection,"drop all objects")
execute(connection, create_schema)
connection.commit()

0 rows
0 rows


## Requêtes

### **Q1** Les athlètes (nom, prénom) dont la date de naissance n'est pas renseignée. (2039 lignes).

In [17]:
query="""
select nomath , prenomath
from athlete
where datenaissance is null
"""
cursor=execute(connection,query,show=True)


2039 rows, display only 10 rows
|nomath                 |prenomath        |
|------------------------------------------
|TOFALVI                |Eva              |
|CHOI                   |Heung-Chul       |
|KIM                    |Hyun-Ki          |
|BAJCICAK               |Martin           |
|SHEVCHENKO             |Valentina        |
|UEMURA                 |Aiko             |
|DERYZEMLYA             |Andriy           |
|TOBRELUTS              |Indrek           |
|MATURA                 |Jan              |
|SIMARI BIRKNER         |Macarena         |
|KIRCHGASSER            |Michaela         |


### **Q2** Le nom de sports contenant la chaîne de caractères 'ski' (par exemple, 'Ski de fond', 'Saut à ski', …). (4 lignes).

In [18]:
query="""
select nomsp
from sport 
where nomsp like '%ski%' or nomsp like '%Ski%'
"""
# or 
# lower(nomsp) like '%ski%'
cursor=execute(connection,query,show=True)

4 rows, display only 10 rows
|nomsp          |
|----------------
|Saut à ski     |
|Ski acrobatique|
|Ski alpin      |
|Ski de fond    |


### **Q3** La nationalité des athlètes Kamil STOCH et Suk-Hee SHIM.(2 lignes).

In [19]:
query="""
select codepays
from athlete
where (nomath = 'STOCH' and prenomath = 'Kamil') or (nomath = 'SHIM' and prenomath = 'Suk-Hee')
"""
#or 
# nomath in ('STOCH','SHIM') and prenomath in ('Kamil','Suk-Hee')
cursor=execute(connection,query,show=True)

2 rows, display only 10 rows
|codepays|
|---------
|POL     |
|KOR     |


### **Q4** Le nom des athlètes originaire de Scandinavie (pays : Danemark, Finlande, Norvège, Suède, Islande) triés par le nom, puis par leur prénom (9724 lignes).

In [20]:
query="""
select a.nomath
from athlete a , pays p 
where a.codepays = p.codepays
and nomp in ('Danemark','Finlande','Norvège','Suède','Islande')
order by nomath,prenomath
"""

cursor=execute(connection,query,show=True)

267 rows, display only 10 rows
|nomath           |
|------------------
|AHONEN           |
|ANDERSSON        |
|ARWIDSON         |
|ASGEIRSDOTTIR    |
|AURDAL           |
|Aaltonen         |
|Alfredsson       |
|BAECK            |
|BARDAL           |
|BERGER           |
|BERGMAN          |


### **Q5** Les épreuves (sport, épreuve et categorie) qui ont eu lieu après le 21 février 2014 triées par nom de sport, puis par nom d'épreuve dans l'ordre inverse du dictionnaire (6 lignes).

In [21]:
query="""
select e.nomep , s.nomsp , e.categorie
from sport s , epreuve e
where e.sid = s.sid
and e.datedebut >  '2014-02-21'
order by s.nomsp, e.nomep Desc
"""

cursor=execute(connection,query,show=True)

6 rows, display only 10 rows
|nomep           |nomsp          |categorie|
|-------------------------------------------
|relais 4x7,5km  |Biathlon       |Hommes   |
|bob à quatres   |Bobsleigh      |Hommes   |
|Slalom          |Ski alpin      |Femmes   |
|50km            |Ski de fond    |Hommes   |
|Slalom parallèle|Surf des neiges|Femmes   |
|Slalom parallèle|Surf des neiges|Hommes   |


### **Q6** Les épreuves (sport, épreuve et catégorie) dont le final a eu lieu pendant un weekend. (34 lignes).

In [22]:
query="""
select s.nomsp ,e.nomep, e.categorie
from sport s , epreuve e
where e.sid = s.sid
and dayofweek(e.datefin) in (1,7)
"""
#  dayname(e.datefin) in ('Saturday', 'Sunday')
cursor=execute(connection,query,show=True)

34 rows, display only 10 rows
|nomsp                               |nomep                     |categorie|
|--------------------------------------------------------------------------
|Biathlon                            |7,5km                     |Femmes   |
|Biathlon                            |10km                      |Hommes   |
|Biathlon                            |relais 4x7,5km            |Hommes   |
|Bobsleigh                           |bob à quatres             |Hommes   |
|Hockey sur glace                    |hockey sur glace          |Hommes   |
|Luge                                |Simple                    |Hommes   |
|Patinage artistique                 |Par équipes               |Mixte    |
|Patinage de vitesse                 |1500m                     |Femmes   |
|Patinage de vitesse                 |3000m                     |Femmes   |
|Patinage de vitesse                 |Poursuite par équipes     |Femmes   |
|Patinage de vitesse                 |1500m               

### **Q7** Les athlètes (nom, prénom) qui ont aujourd'hui moins de 25 ans ayant participé à au moins une épreuve individuelle et au moins une par équipe. Exprimer la requête de trois façons différentes. (dépend de la date d'excution; aujourdhui: 0 lignes) 

In [23]:
query="""
select a.nomath, a.prenomath
from athlete a , athletesequipe aq , rangindividuel ri , rangequipe rq
where a.aid = ri.aid and aq.aid = a.aid and rq.eqid = aq.eqid and datediff('year',a.datenaissance,current_date) < 25
"""
# le plus jeune a 31 en 2026 :< 0  moins de 25 
cursor=execute(connection,query,show=True)

0 rows


### **Q8** Les athlètes ayant gagné une médaille dans une épreuve individuelle le jour de leur anniversaire (la date de fin de l'épreuve coïncide  avec leur anniversaire). (1 ligne).

In [24]:
query="""
select a.*
from athlete a, rangindividuel ri, epreuve e
where a.aid = ri.aid and ri.rang <= 3 and e.epid = ri.epid
and dayofmonth(a.datenaissance) = dayofmonth(e.datefin) and month(a.datenaissance) = month(e.datefin)
"""
cursor=execute(connection,query,show=True)

1 rows, display only 10 rows
|aid|nomath    |prenomath|datenaissance|codepays|
|------------------------------------------------
|382|GARANICHEV|Evgeniy  |1988-02-13   |RUS     |


### **Q9** Les épreuves (sport, épreuve, catégorie) auxquelles participent des  équipes (14 lignes).

In [25]:
query="""
select distinct s.nomsp, e.nomep,e.categorie
from  sport s, epreuve e , rangequipe rq
where e.sid = s.sid and e.epid = rq.epid 


"""

cursor=execute(connection,query,show=True)

25 rows, display only 10 rows
|nomsp                               |nomep                   |categorie|
|------------------------------------------------------------------------
|Biathlon                            |Relais mix              |Mixte    |
|Biathlon                            |relais 4x6km            |Femmes   |
|Biathlon                            |relais 4x7,5km          |Hommes   |
|Bobsleigh                           |bob à deux              |Femmes   |
|Bobsleigh                           |bob à deux              |Hommes   |
|Bobsleigh                           |bob à quatres           |Hommes   |
|Combiné nordique                    |Par équipes             |Hommes   |
|Curling                             |curling                 |Femmes   |
|Curling                             |curling                 |Hommes   |
|Hockey sur glace                    |hockey sur glace        |Femmes   |
|Hockey sur glace                    |hockey sur glace        |Hommes   |


### **Q10** Le pays qui a gagné, en équipe, la médaille d'or dans l'épreuve de la catégorie  'Femmes' intitulée 'relais 4x6km' du sport 'Biathlon' (Ukraine).

In [26]:
query="""
select p.nomp
from pays p, rangequipe rq , equipe e , epreuve ep, sport s
where p.codepays = e.codepays and  rq.eqid = e.eqid and rq.epid = ep.epid
and s.sid = ep.sid  and ep.categorie='Femmes' and s.nomsp = 'Biathlon' and rq.rang = 1
and ep.nomep = 'relais 4x6km'
"""

cursor=execute(connection,query,show=True)

1 rows, display only 10 rows
|nomp   |
|--------
|Ukraine|


### **Q11** Les homonymes (les nom de familles portés par deux athlètes ou plus)(141 lignes).

In [27]:
query="""
select distinct a1.nomath
from athlete a1 
where exists (select *
            from athlete a2 
            where a2.aid != a1.aid and a1.nomath = a2.nomath)
"""
# a1.nomath in  ( select a2.nomath ........)        #1
# where a1.nomath = a2.nomath and a1.aid != a2.aid  #2
# group by nomath having count(*) >= 2 TD7          #3
cursor=execute(connection,query,show=True)

141 rows, display only 10 rows
|nomath         |
|----------------
|ABRAMENKO      |
|ANDERSON       |
|BAUER          |
|BAUMANN        |
|BECKERT        |
|BJORNSEN       |
|BOE            |
|BOWMAN         |
|BRAATEN        |
|BROWN          |
|BROZ           |


### **Q12** Les athlètes ayant gagné une médaille dans une épreuve individuelle, mais ayant été disqualifiés dans une autre. (14 lignes).

In [28]:
query="""


"""

cursor=execute(connection,query,show=True)

0 rows


### **Q13** Les sports qui n'ont pas d'épreuves féminines. (Résultat :  Combiné nordique)

In [29]:
query=""" 
select s.nomsp
from sport s
where s.sid not in (select e.sid
                  from epreuve e 
                  where e.categorie = 'Femmes')
"""
#where not exists (select * from epreuve e 
#                  where e.sid = s.sid and e.categorie = 'Femmes')  !! 'Femmes' != 'Femme'
cursor=execute(connection,query,show=True)

1 rows, display only 10 rows
|nomsp           |
|-----------------
|Combiné nordique|


### **Q14** Les noms de pays qui n’ont qu’une seule équipe. (9 lignes)

In [30]:
query=""" 
select p.nomp
from pays p , equipe e 
where e.codepays  = p.codepays
and not exists (select * 
                from equipe e2
                where e.eqid != e2.eqid and e.codepays = e2.codepays)
"""
# group by p.nom  having count(*) =1 TD7
cursor=execute(connection,query,show=True)

9 rows, display only 10 rows
|nomp       |
|------------
|Azerbaïdjan|
|Belgique   |
|Bulgarie   |
|Espagne    |
|Jamaïque   |
|Lituanie   |
|Monaco     |
|Serbie     |
|Turquie    |


### **Q15** Les athlètes ayant participé à exactement 2 épreuves individuelles.

In [31]:
query="""
select distinct a.nomath, a.prenomath
from athlete a, rangindividuel r1, rangindividuel r2
where r1.aid = a.aid
and r2.aid = a.aid
and r1.epid <> r2.epid
and not exists (
    select *
    from rangindividuel r3
    where r3.aid = a.aid
    and r3.epid <> r1.epid
    and r3.epid <> r2.epid
)
"""

cursor=execute(connection,query,show=True)

402 rows, display only 10 rows
|nomath           |prenomath        |
|------------------------------------
|ABDERHALDEN      |Marianne         |
|ABRAMASHVILI     |Iason            |
|ACHIRILOAIE      |Ioan Valeriu     |
|AGREITER         |Debora           |
|AHONEN           |Janne            |
|AKHMADIYEV       |Yerdos           |
|ALEXANDER        |Nicholas         |
|ALMOUKOV         |Alexei           |
|AMMANN           |Simon            |
|ANDERSSON        |David            |
|ANGERMUELLER     |Monique          |


# TME5: JOINTURES ET IMBRICATION AVEC IN ET EXISTS 

Ce TME utilise les données contenues dans le fichier **bd_jo_v2_H2.sql**


## Requêtes

### **Q1**. Les athlètes nés entre 1990 et 1995. (Ski de fond, Biathlon).
Résultat : (87 lignes)

In [32]:
query="""
select a.aid 
from athlete a 
where year(a.datenaissance) >= '1990' and year(a.datenaissance) <= 1995
"""

cursor=execute(connection,query,show=True)

87 rows, display only 10 rows
|aid|
|----
|17 |
|34 |
|47 |
|56 |
|72 |
|73 |
|74 |
|82 |
|94 |
|120|
|123|


### **Q2**. Les épreuves (sport, épreuve, catégorie) individuelles.
Résultat : (73 lignes)

In [33]:
query="""
select distinct s.nomsp , e.nomep , e.categorie
from sport s, epreuve e , rangindividuel r
where s.sid = e.sid and r.epid = e.epid
"""

cursor=execute(connection,query,show=True)

73 rows, display only 10 rows
|nomsp                               |nomep                     |categorie|
|--------------------------------------------------------------------------
|Biathlon                            |10km                      |Hommes   |
|Biathlon                            |10km poursuite            |Femmes   |
|Biathlon                            |12,5km départ groupé      |Femmes   |
|Biathlon                            |12,5km poursuite          |Hommes   |
|Biathlon                            |15km                      |Femmes   |
|Biathlon                            |15km départ groupé        |Hommes   |
|Biathlon                            |20km                      |Hommes   |
|Biathlon                            |7,5km                     |Femmes   |
|Combiné nordique                    |Individuel                |Hommes   |
|Combiné nordique                    |Individuel LH             |Hommes   |
|Luge                                |Simple              

### **Q3**. Les résultats (nom, prénom, pays de l'athlète et rang) de l'épreuve '1000m' du 'Patinage de vitesse' pour les 'Femmes'.
Résultat : (36 lignes)

In [34]:
query="""
select a.nomath, a.prenomath, a.codepays , r.rang
from athlete a , rangindividuel r , epreuve e, sport s
where a.aid  = r.aid and r.epid  = e.epid and s.sid = e.sid
and e.nomep like '%1000m%' and s.nomsp ='Patinage de vitesse' and  e.categorie = 'Femmes'
"""

cursor=execute(connection,query,show=True)

36 rows, display only 10 rows
|nomath       |prenomath |codepays|rang|
|---------------------------------------
|HONG         |Zhang     |CHN     |1   |
|WÜST         |Ireen     |NED     |2   |
|BOER         |Margot    |NED     |3   |
|FATKULINA    |Olga      |RUS     |4   |
|BEEK         |Lotte van |NED     |5   |
|LEENSTRA     |Marrit    |NED     |6   |
|RICHARDSON   |Heather   |USA     |7   |
|BOWE         |Brittany  |USA     |8   |
|NESBITT      |Christine |CAN     |9   |
|ERBANOVA     |Karolina  |CZE     |10  |
|HESSE        |Judith    |GER     |11  |


### **Q4**. Le nom et prénom des athlètes qui forment l'équipe qui a gagné la médaille d'or dans l'épreuve 'relais 4x6km' de 'Biathlon' de 'Femmes'. 
Résultat : SEMERENKO Vita, SEMERENKO Valj, DZHYMA Juliya, PIDHRUSHNA Olena

In [35]:
query="""
select distinct a.nomath , a.prenomath
from athlete a , rangequipe rq , athletesequipe aq , epreuve e, sport s
where a.aid = aq.aid and rq.eqid  = aq.eqid and  e.sid = s.sid and rq.epid  =e.epid 
and e.categorie = 'Femmes' and s.nomsp = 'Biathlon' and e.nomep  = 'relais 4x6km' and rq.rang = 1
""" 

cursor=execute(connection,query,show=True)

4 rows, display only 10 rows
|nomath    |prenomath|
|---------------------
|DZHYMA    |Juliya   |
|PIDHRUSHNA|Olena    |
|SEMERENKO |Valj     |
|SEMERENKO |Vita     |


### **Q5**. Les sports auxquels LESSER Erik a participé en supposant qu'il a participé aux épreuves
individuelles
Résultat: (Ski de fond, Biathlon).

In [36]:
query="""
select distinct s.nomsp
from sport s , epreuve e, rangindividuel ri,athlete a
where a.aid = ri.aid and e.epid = ri.epid and s.sid = e.sid and a.nomath ='LESSER' and a.prenomath= 'Erik'
"""

cursor=execute(connection,query,show=True)

2 rows, display only 10 rows
|nomsp      |
|------------
|Biathlon   |
|Ski de fond|


### **Q6**. Les athlètes suisses (nom, prenom) ayant participé au sport 'Biathlon' et disqualifié à au moins une épreuve en individuel de ce sport.
Résultat : GASPARIN Elisa

In [37]:
query="""
select distinct a.nomath, a.prenomath , a.codepays
from athlete a , epreuve e , rangindividuel r1, sport s , pays p
where a.aid = r1.aid and e.epid = r1.epid and s.sid = e.sid 
and s.nomsp = 'Biathlon' and r1.rang is not null and a.codepays = p.codepays and a.codepays= 'SUI'
and exists (select * 
            from rangindividuel r2
            where r2.aid = a.aid and r2.epid != r1.epid and r2.rang is null )
"""

cursor=execute(connection,query,show=True)

1 rows, display only 10 rows
|nomath  |prenomath|codepays|
|----------------------------
|GASPARIN|Elisa    |SUI     |


### **Q7**. Les athlètes ayant participé aux épreuves individuelles de (au moins) deux sports différents ( PETROVIC Milanko, TER MORS Jorien, LESSER Erik, PEIFFER Arnd).
Résultat : PETROVIC Milanko, TER MORS Jorien, LESSER Erik, PEIFFER Arnd

In [38]:
query="""
select distinct a.nomath, a.prenomath
from athlete a, rangindividuel r1 , epreuve e, sport s
where a.aid  = r1.aid and r1.epid= e.epid and s.sid = e.sid 
and exists (select *
            from rangindividuel r2 ,epreuve e2
            where r2.aid = a.aid and r2.epid != r1.epid and r2.epid = e2.epid and e2.sid != s.sid )
"""

cursor=execute(connection,query,show=True)

4 rows, display only 10 rows
|nomath  |prenomath|
|-------------------
|LESSER  |Erik     |
|PEIFFER |Arnd     |
|PETROVIC|Milanko  |
|TER MORS|Jorien   |


### **Q8**. Les dates de début et de fin des Jeux Olympique 2014 (06/02/2014, 23/02/2014).
Résultat : 06/02/2014 – 23/02/2014

In [58]:
query="""
select distinct e.datedebut, e3.datefin
from epreuve e , epreuve e3
where not exists (select *
                    from epreuve  e2 
                    where e2.datedebut < e.datedebut)
and not exists ( select *
                from epreuve e4 
                where e4.datefin > e3.datefin)
"""

cursor=execute(connection,query,show=True)

1 rows, display only 10 rows
|datedebut |datefin   |
|----------------------
|2014-02-06|2014-02-23|


### **Q9**. Les pays qui ont des participants de moins de 18 ans ou de plus que 40 au 24-02-2016
(Attention : il faut éliminer les athlètes dont on ne connaît pas la date de naissance) 
Résultat :  (9 lignes)

In [101]:
query="""
select distinct a.codepays
FROM   athlete a
where a.datenaissance is not null 
and ( datediff('year',a.datenaissance, '2016-02-24') <= 18 or datediff('year' , a.datenaissance, '2016-02-24') >= 40)
"""

cursor=execute(connection,query,show=True)

9 rows, display only 10 rows
|codepays|
|---------
|AUT     |
|CAN     |
|GER     |
|ITA     |
|JPN     |
|NED     |
|NOR     |
|RUS     |
|USA     |


### **Q10**. Les équipes n'ayant pas d'athlète (la base de données ne contient pas l'information sur les participants)
Résultat: (13 lignes)

In [41]:
query="""
select e.*
from equipe e
where not exists(select *
                  from athletesequipe a
                   where a.aid is not null and a.eqid = e.eqid)
"""
# e.eqid  not in ( ... )
# e.eqid != any ( ... )
cursor=execute(connection,query,show=True)

13 rows, display only 10 rows
|eqid|codepays|
|--------------
|232 |CHN     |
|160 |FIN     |
|231 |FRA     |
|235 |GBR     |
|162 |GER     |
|233 |GER     |
|229 |ITA     |
|163 |JPN     |
|230 |JPN     |
|161 |RUS     |
|159 |SWE     |


### **Q11**. Les épreuves (nom sport, nom epreuve, categorie) dans lesquelles il n'y a pas eu une médaille d'argent en individuel.
Résultat : (26 lignes)

In [42]:
query="""
select s.nomsp , e.nomep ,e.categorie 
from sport s, epreuve e
where s.sid = e.sid
and not exists (select *
                from rangindividuel ri
                where ri.epid = e.epid and ri.rang =2)
"""
# e.epid  not in ( ... )
# e.epid != any ( ... )
cursor=execute(connection,query,show=True)

26 rows, display only 10 rows
|nomsp                               |nomep                   |categorie|
|------------------------------------------------------------------------
|Biathlon                            |relais 4x6km            |Femmes   |
|Biathlon                            |relais 4x7,5km          |Hommes   |
|Biathlon                            |Relais mix              |Mixte    |
|Bobsleigh                           |bob à deux              |Femmes   |
|Bobsleigh                           |bob à deux              |Hommes   |
|Bobsleigh                           |bob à quatres           |Hommes   |
|Combiné nordique                    |Par équipes             |Hommes   |
|Curling                             |curling                 |Femmes   |
|Curling                             |curling                 |Hommes   |
|Hockey sur glace                    |hockey sur glace        |Femmes   |
|Hockey sur glace                    |hockey sur glace        |Hommes   |


### **Q12**. Les équipes qui ont exactement 2 athlètes. Retourner l'équipe en question avec ses deux seuls athlètes
Résultat: (114 lignes).


In [66]:
query="""
select e.eqid , a1.nomath ,a2.nomath
from equipe e , athlete a1, athlete a2, athletesequipe aq, athletesequipe aq2
where e.eqid = aq.eqid and e.eqid = aq2.eqid and a1.aid = aq.aid and a2.aid = aq2.aid and a1.aid < a2.aid  
and not exists (select *
                from athletesequipe aq3 
                where aq3.eqid = e.eqid and aq3.aid not in ( a1.aid ,a2.aid))
"""
# a1.aid < a2.aid !!
# group by eqid having count(distinct a.aid ) = 2
cursor=execute(connection,query,show=True)

114 rows, display only 10 rows
|eqid|nomath         |nomath         |
|-------------------------------------
|65  |RAWLINSON      |RADJENOVIC     |
|96  |HARVEY         |SPENCE         |
|221 |OBRIEN         |MERRIMAN       |
|66  |HENGSTER       |TUECHI         |
|92  |MAIER          |SAMMER         |
|169 |LINGER         |LINGER         |
|186 |FISCHLER       |PENZ           |
|283 |SMUTNA         |STADLOBER      |
|213 |SITNIKOV       |ZLOBINA        |
|57  |WILLEMSEN      |Emilie MARIEN  |
|70  |SANTOS         |Mayara DA SILVA|


### **Q13**. Les sports qui n'ont pas d'épreuves de catégorie Mixte 
Résultat: (12 lignes).

In [67]:
query="""
select s.*
from sport s
where not exists (select * 
                    from epreuve e 
                    where e.sid = s.sid and e.categorie ='Mixte')
"""

cursor=execute(connection,query,show=True)

12 rows, display only 10 rows
|sid|nomsp                               |
|-----------------------------------------
|2  |Bobsleigh                           |
|3  |Combiné nordique                    |
|4  |Curling                             |
|5  |Hockey sur glace                    |
|8  |Patinage de vitesse                 |
|9  |Patinage de vitesse sur piste courte|
|10 |Saut à ski                          |
|11 |Skeleton                            |
|12 |Ski acrobatique                     |
|13 |Ski alpin                           |
|14 |Ski de fond                         |


### **Q14**. Les athlètes qui ont gagné une médaille d’or (au moins) mais pas de médaille d’argent ni de bronze
Résultat: (53 lignes)

In [70]:
query="""
select distinct a.*
from athlete a , rangindividuel ri
where a.aid = ri.aid and ri.rang = 1
and not exists (select * 
                from rangindividuel r2
                where r2.aid = a.aid and r2.rang in (2,3) )
"""

cursor=execute(connection,query,show=True)

53 rows, display only 10 rows
|aid |nomath         |prenomath       |datenaissance|codepays|
|-------------------------------------------------------------
|1   |BJOERNDALEN    |Ole Einar       |1974-01-27   |NOR     |
|2   |BJOERGEN       |Marit           |1980-03-21   |NOR     |
|6   |SVENDSEN       |Emil Hegle      |1985-07-12   |NOR     |
|10  |HAMELIN        |Charles         |1984-04-14   |CAN     |
|14  |DOMRACHEVA     |Darya           |1986-08-03   |BLR     |
|15  |COLOGNA        |Dario           |1986-03-11   |SUI     |
|16  |LOCH           |Felix           |1989-07-24   |GER     |
|17  |ZHOU           |Yang            |1991-06-09   |CHN     |
|23  |MAZE           |Tina            |1983-05-02   |SLO     |
|28  |KOWALCZYK      |Justyna         |1983-01-23   |POL     |
|39  |GEISENBERGER   |Natalie         |1988-02-05   |GER     |


### **Q15**. Les athlète(s) qui ont fini dernier d’une épreuve individuelle. Indiquer leur nom, prenom, le nom du sport et de l’épreuve et leur rang. Attention : il faut filtrer les athlètes disqualifiés 
Résultat: (73 lignes).

In [79]:
query="""
select distinct a.nomath, a.prenomath, s.nomsp ,e.nomep ,ri.rang
from sport s, athlete a, epreuve e , rangindividuel ri
where a.aid = ri.aid and e.sid = s.sid and ri.epid = e.epid and ri.rang is not null
and not exists (select *
                from rangindividuel r2
                where r2.epid = e.epid and r2.aid != a.aid and r2.rang > ri.rang)
"""

cursor=execute(connection,query,show=True)

73 rows, display only 10 rows
|nomath       |prenomath      |nomsp                               |nomep                     |rang|
|---------------------------------------------------------------------------------------------------
|ABOLTINA     |Agnese         |Ski alpin                           |Super G                   |31  |
|ALEXANDER    |Nicholas       |Saut à ski                          |Grand tremplin individuel |48  |
|BERECZ       |Anna           |Ski alpin                           |Descente                  |35  |
|BROCKHOFF    |Belle          |Surf des neiges                     |Snowboard cross           |8   |
|CAILL        |Ania Monica    |Ski alpin                           |Super combiné             |22  |
|CALDWELL     |Sophie         |Ski de fond                         |Sprint 1,5km              |6   |
|CARCELEN     |Roberto        |Ski de fond                         |15km                      |87  |
|CETINKAYA    |Kelime         |Ski de fond                   

# Fermer la connexion

In [ ]:
connection.commit() # implicit avec close
connection.close()

In [ ]:
connection